# Practical 9 — Email Spam Classification using KNN and SVM

Standalone solution notebook for **ML Practical 9**. Run cells from top to bottom and place the required dataset CSV in the working directory.https://www.kaggle.com/datasets/balaka18/email-spam-classification-dataset-csv

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, mean_squared_error, mean_absolute_error,
    r2_score, roc_auc_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

---
# Practical 9 — Email Spam Classification using KNN and SVM
**Aim:** Classify emails as spam (`1`) or not spam (`0`) and compare K-Nearest Neighbors with Support Vector Machine.

Expected dataset: commonly `emails.csv`, with an email/name column, word-count feature columns, and a label column such as `Prediction`, `spam`, or `label`.

In [ ]:
SPAM_PATH = "emails.csv"
if not os.path.exists(SPAM_PATH):
    alternatives = ["email.csv", "spam.csv", "Email_spam.csv"]
    found = next((f for f in alternatives if os.path.exists(f)), None)
    if found:
        SPAM_PATH = found
    else:
        raise FileNotFoundError(
            "Email spam dataset not found. Put the CSV beside this notebook and set SPAM_PATH."
        )

spam_df = pd.read_csv(SPAM_PATH)
print("Shape:", spam_df.shape)
display(spam_df.head())
print("Columns:", spam_df.columns.tolist()[-10:])

In [ ]:
# Detect label column; edit LABEL_COL manually if your dataset uses another name.
label_candidates = ["prediction", "label", "spam", "target", "class", "is_spam"]
lower_to_original = {str(c).strip().lower(): c for c in spam_df.columns}
LABEL_COL = next((lower_to_original[c] for c in label_candidates if c in lower_to_original), None)

# Common Kaggle version uses a final column named Prediction.
if LABEL_COL is None:
    possible = [c for c in spam_df.columns if spam_df[c].nunique(dropna=True) <= 2]
    if possible:
        LABEL_COL = possible[-1]
if LABEL_COL is None:
    raise ValueError("Could not identify the spam label column. Set LABEL_COL to the correct column name.")

y = spam_df[LABEL_COL]
# Convert textual labels to 0/1 if necessary.
if not pd.api.types.is_numeric_dtype(y):
    y_norm = y.astype(str).str.strip().str.lower()
    mapping = {"spam": 1, "not spam": 0, "ham": 0, "yes": 1, "no": 0, "true": 1, "false": 0}
    y = y_norm.map(mapping)
    if y.isna().any():
        raise ValueError("Label values are not recognized. Convert the label column to 0/1.")

X = spam_df.drop(columns=[LABEL_COL]).copy()
# Remove identifier/text columns that are not numeric word-count features.
X = X.drop(columns=[c for c in X.columns if str(c).strip().lower() in
                    ["email name", "email", "filename", "name", "unnamed: 0"]], errors="ignore")
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# Scale sparse/high-dimensional word-count features for distance and margin-based models.
X_train, X_test, y_train, y_test = train_test_split(
    X, y.astype(int), test_size=0.2, random_state=RANDOM_STATE, stratify=y.astype(int)
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Feature shape:", X.shape, "| label column:", LABEL_COL)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

spam_models = {
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Linear SVM": LinearSVC(random_state=RANDOM_STATE, max_iter=5000)
}
spam_results = []

for name, model in spam_models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    spam_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": __import__("sklearn.metrics").metrics.f1_score(y_test, pred, zero_division=0)
    })
    print("\n", name)
    print(classification_report(y_test, pred, zero_division=0))
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

display(pd.DataFrame(spam_results).sort_values("F1", ascending=False))

**Result:** KNN and Linear SVM are trained and evaluated with accuracy, precision, recall, F1-score, and confusion matrices. For the large 3,000-word dataset, Linear SVM is usually more computationally practical than a distance-based KNN model.